In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

with (project_root / "models" / "modeling_config.json").open(
    encoding="utf-8"
) as file:
    modeling_config = json.load(file)

selected_pipeline = joblib.load(
    project_root / "models" / "logistic_with_campaign_type.joblib"
)

modeling_table = pd.read_parquet(
    project_root / "data" / "processed"
    / "household_campaign_modeling.parquet"
)

print("Selected logistic regression pipeline loaded.")
print(f"Modeling examples: {len(modeling_table):,}")

Selected logistic regression pipeline loaded.
Modeling examples: 7,091


In [2]:
test_data = modeling_table.loc[
    modeling_table["START_DAY"] >= modeling_config["test_start_day"]
].copy()

test_feature_columns = modeling_config["campaign_type_features"]

X_test = test_data[test_feature_columns].copy()
y_test = test_data[modeling_config["target"]].copy()

assert list(selected_pipeline.feature_names_in_) == test_feature_columns

print(f"Test input shape: {X_test.shape}")
print(f"Test campaigns: {test_data['CAMPAIGN'].nunique()}")
print("Feature names and order match the saved pipeline.")

Test input shape: (2218, 7)
Test campaigns: 7
Feature names and order match the saved pipeline.


In [3]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

test_probabilities = selected_pipeline.predict_proba(X_test)[:, 1]

training_redemption_rate = modeling_table.loc[
    modeling_table["END_DAY"] < modeling_config["validation_start_day"],
    modeling_config["target"]
].mean()

constant_test_probabilities = np.full(
    len(y_test),
    training_redemption_rate
)

test_metrics = {
    "test_redemption_rate": y_test.mean(),
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "average_precision": average_precision_score(y_test, test_probabilities),
    "brier_score": brier_score_loss(y_test, test_probabilities),
    "constant_baseline_brier": brier_score_loss(
        y_test, constant_test_probabilities
    )
}

pd.Series(test_metrics).round(4)

test_redemption_rate       0.1429
roc_auc                    0.6804
average_precision          0.2579
brier_score                0.1191
constant_baseline_brier    0.1239
dtype: float64

In [4]:
evaluation_folder = project_root / "reports" / "model_evaluation"

pd.DataFrame([test_metrics]).to_csv(
    evaluation_folder / "final_test_metrics.csv",
    index=False
)

test_predictions = test_data[
    ["household_key", "CAMPAIGN", "START_DAY", "END_DAY", "redeemed"]
].copy()

test_predictions["predicted_redemption_probability"] = test_probabilities

test_predictions.to_csv(
    evaluation_folder / "final_test_predictions.csv",
    index=False
)

print("Saved final test metrics.")
print(f"Saved predictions for {len(test_predictions):,} examples.")

Saved final test metrics.
Saved predictions for 2,218 examples.
